# 02 Metrics for Filter Pairs

This notebook demonstrates and explains `maf.FilterPairTGapsMetric`, a MAF (Metric Analysis Framework)
metric that evaluates how well a given LSST cadence/OpSim run samples **color information** for
time-variable and transient sources.

## What does the filter-pair time-gap metric actually measure?

Rubin/LSST observes the sky in six bands: u, g, r, i, z, y. To measure the **color** of a variable or
transient source -- and how that color evolves -- you need at least two observations taken in
**different filters**, close enough together in time that the source has not changed too much between
them. For any two visits to the same field taken in two different bands (a "filter pair", e.g. g-r,
r-i, u-g, ...), the time interval between them is called a **filter-pair time gap** ("tgap").

Conceptually, for each sky position (each Healpix pixel) the metric:

1. collects all the visits recorded there over the selected time range;
2. considers every pair of visits taken in two *different* bands;
3. computes the time gap (Delta t) between the two visits making up each pair;
4. summarizes these gaps into one (or a few) representative value(s) for that pixel.

This is a direct implementation of the **filterTGapsMetric** concept introduced by Li et al. (2021, AJ),
"Preparing to discover the unknown with Rubin LSST -- I: Time domain" (arXiv:2107.10281). That paper
developed this metric to assess LSST's ability to discover both known and *unanticipated* fast-evolving
transients and variable phenomena via combined color + time coverage:

- **Short gaps** (minutes up to about 1.5 hours) are what you need to measure a reliable, nearly
  "instantaneous" color for fast transients (e.g. kilonovae, fast blue optical transients, shock
  breakout, the earliest hours of a supernova).
- **A broad, well-populated spread of gaps** (from minutes to many days) indicates a cadence that also
  samples color *evolution* across many timescales -- valuable for a wide range of slower phenomena.

**Why this matters**: the classic LSST "visit pairs" (two back-to-back visits ~20-30 minutes apart,
used for moving-object / asteroid detection) are normally taken in the *same* filter. Same-filter pairs
only tell you how bright something is and how fast it is fading -- they carry no color information at
all. Only a pair of visits in two *different* filters lets you measure a color at (nearly) a single
epoch. So the number and time-distribution of filter-pair gaps at a given sky position is a direct proxy
for how well that cadence supports multi-color transient science there.

**A note on precision**: the exact algorithm, input parameters, and returned value(s) of
`maf.FilterPairTGapsMetric` can evolve between `rubin_sim` releases, and may differ in detail from the
original 2021 paper (which computed a Kullback-Leibler-divergence-based Figure of Merit; the packaged
MAF metric may implement a simpler/updated summary). The explanation above describes the general,
published methodology this metric is built on. To see exactly what *your* installed version does, run
the introspection cells in the next section -- they print the metric's own docstring and source code.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rubin_sim.maf as maf  # rubin_sim.maf = the Metric Analysis Framework (MAF) used to evaluate an LSST cadence/OpSim run

try:
    from rubin_sim.data import get_baseline  # path to the baseline OpSim database (older rubin_sim layout)
except ImportError:
    from rubin_scheduler.data import get_baseline  # newer rubin_scheduler layout

## Configuration

In [ ]:
opsdb_fname = get_baseline()  # path to the baseline OpSim sqlite database (the simulated cadence to analyze)
run_name = os.path.split(opsdb_fname)[-1].replace(
    ".db", ""
)  # short label for this run, used in plot titles/legends
print(f"Using {run_name}, to be read from {opsdb_fname}")

In [ ]:
data_dir = None

if data_dir is None:
    # no explicit output directory given -> create a scratch temporary directory for MAF's intermediate
    # output (sqlite results database, cached npz files, etc.); it is cleaned up when the notebook/kernel exits
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="02_filterpairs_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

## Inspect the metric itself (optional, but recommended)

The two cells below use IPython's introspection magics to show you exactly what is implemented in your
installed `rubin_sim` version:

- `%pinfo` prints the metric's docstring: its parameters, defaults, and a description of what it returns.
- `%psource` prints the full Python source code of the metric class.

Run these once to confirm the precise behavior (units, default filter pairs considered, what exactly is
returned per pixel) before trusting the conceptual explanation above in detail.

In [ ]:
%pinfo maf.FilterPairTGapsMetric

In [ ]:
%psource maf.FilterPairTGapsMetric

## Define the slicer and the metric

In [ ]:
# Look at the filter pairs metric for random discovery
# FilterPairTGapsMetric: for each Healpix pixel, looks at all pairs of visits taken in two different
# bands and summarizes their time gaps (see the introspection cells above for the precise definition
# and units used by your installed version).
# filter_col="band": tells the metric which OpSim database column holds the filter/band name for each
# visit. Newer OpSim schemas use "band"; older ones use "filter" -- if this raises a column-not-found
# error, try filter_col="filter" instead.
m = maf.FilterPairTGapsMetric(filter_col="band")

# HealpixSlicer: evaluates the metric independently at every Healpix pixel on the sky (nside=64 sets the
# pixel resolution), turning the per-field result into a full-sky map.
s = maf.HealpixSlicer(nside=64)

# plotDict: bounds/appearance for the sky map and histogram plots produced later by bundle.plot().
# The x/color ranges below (0-1500/2000) are presumably in minutes given the metric's short-timescale
# color-science motivation -- confirm the exact units from the docstring above.
# NOTE (bug fix): newer rubin_sim releases use snake_case plot_dict keys (x_min/x_max/color_min/
# color_max, ...) rather than the older camelCase (xMin/xMax/colorMin/colorMax). Passing the old
# camelCase keys does NOT raise an error -- unknown dict keys are silently ignored -- so the plot
# would silently fall back to auto-scaling from the data instead of respecting these bounds. See the
# same issue (and fix) documented in 02_TDC_TimeDelayAccuracy.ipynb.
plotDict = {"color_min": 0, "color_max": 1500, "x_min": 0, "x_max": 2000}

# summary_metrics: reduce the whole-sky map of per-pixel metric values down to single numbers.
summarystats = [
    maf.MedianMetric(),  # sky-median value: the typical (robust) filter-pair time gap across the footprint
    maf.MeanMetric(),  # sky-averaged value: more sensitive to outlier pixels (e.g. very sparsely observed fields)
    maf.PercentileMetric(
        percentile=80
    ),  # value below which 80% of sky pixels fall: a tail/robustness indicator
]

# MetricBundle: packages the metric, the slicer, an (absent, here) SQL constraint, and the plotting/summary
# configuration into one object that MetricBundleGroup can execute against the OpSim database.
bundle = maf.MetricBundle(m, s, None, plot_dict=plotDict, summary_metrics=summarystats, run_name=run_name)

## Run the Bundle group

In [ ]:
%%time
# MetricBundleGroup: connects to the OpSim sqlite database, runs the SQL query needed by the bundle(s),
# and evaluates the metric at every slice point (here, every Healpix pixel) defined by the slicer.
# {"0": bundle}: a dict of bundles to run together, keyed by an arbitrary label ("0" here, since we only
# have a single bundle in this notebook).
g = maf.MetricBundleGroup({"0": bundle}, opsdb_fname, data_dir)
g.run_all()

## Access the per-pixel metric values

`bundle.metric_values` is a numpy masked array with one entry per Healpix pixel: unmasked entries hold
the metric's result for that pixel; masked entries are pixels where the metric could not be computed
(e.g. no visits, or not enough visits in different filters, in that pixel). `.compressed()` returns just
the valid (unmasked) values as a flat 1D array, convenient for statistics or custom plots.

In [ ]:
bundle.metric_values

In [ ]:
bundle.metric_values.compressed()

## Plot

`bundle.plot()` produces the slicer's default plots for this metric: typically a Healpix sky map (showing
how the filter-pair time gap varies across the footprint) and a histogram (showing the distribution of
values over all pixels).

In [ ]:
bundle.plot()

## Summary statistics

`bundle.summary_values` holds the three numbers computed by `summarystats` above: the sky-median,
sky-mean, and 80th-percentile of the filter-pair time-gap metric across all valid pixels. These give a
single-number sense of how well (and how uniformly) this cadence samples multi-color information, useful
for comparing this OpSim run against others (see the next section).

In [ ]:
bundle.summary_values

## Comparison across many OpSim runs

This section pulls MAF's archive of pre-computed summary-metric values for many OpSim runs, so this
cadence's filter-pair time-gap statistics can be compared side by side against other strategies.

**Portability note**: the cell below currently hard-codes `metric_set = maf.get_metric_sets("/Users/lynnej/lsst_repos/survey_strategy/fbs_2.0/metric_sets.json")`,
an absolute path on another user's machine. This will likely fail (or silently use a stale/missing file)
on any other computer, including this one. You have two options:

1. Call `maf.get_metric_sets()` with no argument to use the version's built-in default metric-set
   definitions, or
2. Point it at your own local copy of a `metric_sets.json` file, if you have one.

The cell below has been left using the built-in default to make the notebook portable; adjust if you
specifically need the file referenced above.

In [ ]:
# Let's compare across many runs
# families: MAF's archive of named groups of related OpSim runs (e.g. "baseline", "rolling", "good seeing", ...)
families = maf.archive.get_family_descriptions()
family_list = families.index.values

# summaries: the full table of pre-computed summary-metric values (one row per OpSim run, one column per
# named summary metric) for a given archived summary-statistics release
summary_source = "summary_2022_08_01.csv"
summaries = maf.get_metric_summaries(summary_source=summary_source)

# metric_set: named, curated collections of summary metrics (e.g. grouped by science case) used to drive
# the comparison plots below. Using the built-in default here instead of another user's local file path
# (see the portability note above) keeps this notebook runnable on any machine.
metric_set = maf.get_metric_sets()

In [ ]:
# compare filter gaps pairs metric results with total number of visits

# select, from the full summaries table, only the columns relevant to filter-pair time gaps and overall
# visit counts: the median FilterPairTGaps value, the total WFD visit count (excluding any per-slicer
# breakdowns), and the median TgapsPercent value (the companion "same-night vs next-night" percentage
# metric described in the introduction above)
metrics = [
    m
    for m in summaries
    if "Median FilterPairTGaps" in m
    or ("Nvisits WFD" in m and "Slicer" not in m)
    or ("Median TgapsPercent" in m)
]
mset = maf.create_metric_set_df("test", metrics)

# fams: the OpSim run families to include in the comparison (a representative cross-section of strategies)
fams = [
    "baseline",
    "rolling",
    "triplets",
    "long gaps no pairs",
    "suppress repeats",
    "good seeing",
    "bluer balance",
    "vary nes",
]
these_runs = families.explode("run").loc[fams, "run"]

# plot_run_metric: a per-run comparison plot of the selected metrics, each normalized relative to the
# baseline_run (so a value of 1.0 means "same as baseline"); lets you see at a glance which strategies
# sample filter-pair color information better or worse than the baseline, alongside their raw visit counts
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, metrics], baseline_run="baseline_v2.0_10yrs", metric_set=mset
)
fig.set_figheight(30)
ax.set_xlim(0.7, 1.1)

In [ ]:
# "TVS anomalies" is a curated metric set (defined by the Transient and Variable Stars Science
# Collaboration) that bundles together the summary metrics most relevant for judging a cadence's power to
# discover anomalous/unexpected transients and variables -- including filter-pair time-gap metrics like
# the one in this notebook, alongside related color- and cadence-sensitivity metrics.
mset = metric_set.loc["TVS anomalies"]
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, mset["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    metric_set=mset,
    metric_label_map=mset["short_name"],
    vertical_quantity="value",  # metric values go on the vertical axis
    horizontal_quantity="run",  # OpSim runs are spread along the horizontal axis
)
fig.set_figwidth(20)